In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
!tar xf spark-3.1.1-bin-hadoop3.2.tgz
!pip install -q findspark

Impossibile trovare il percorso specificato.
"wget" non � riconosciuto come comando interno o esterno,
 un programma eseguibile o un file batch.
tar: Error opening archive: Failed to open 'spark-3.1.1-bin-hadoop3.2.tgz'

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"

In [3]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
spark

Exception: Unable to find py4j in /content/spark-3.1.1-bin-hadoop3.2\python, your SPARK_HOME may not be configured correctly

In [4]:
import pyspark
type(spark)

ModuleNotFoundError: No module named 'pyspark'

In [5]:
sc = spark.sparkContext

In [40]:
s = '''Every person had a star, every star had a friend, and for every person
carrying a star there was someone else who reflected it, and everyone
carried this reflection like a secret confidante in the heart'''
simple_rdd = sc.parallelize(s.split('\n'))

In [41]:
simple_rdd

ParallelCollectionRDD[112] at readRDDFromFile at PythonRDD.scala:274

In [42]:
simple_rdd.collect()

['Every person had a star, every star had a friend, and for every person',
 'carrying a star there was someone else who reflected it, and everyone',
 'carried this reflection like a secret confidante in the heart']

In [43]:
simple_rdd.map(lambda line: line.split(' '))

PythonRDD[113] at RDD at PythonRDD.scala:53

In [44]:
(simple_rdd.map(lambda line: line.split(' '))
           .collect())

[['Every',
  'person',
  'had',
  'a',
  'star,',
  'every',
  'star',
  'had',
  'a',
  'friend,',
  'and',
  'for',
  'every',
  'person'],
 ['carrying',
  'a',
  'star',
  'there',
  'was',
  'someone',
  'else',
  'who',
  'reflected',
  'it,',
  'and',
  'everyone'],
 ['carried',
  'this',
  'reflection',
  'like',
  'a',
  'secret',
  'confidante',
  'in',
  'the',
  'heart']]

In [45]:
(simple_rdd.flatMap(lambda line: line.split(' '))
           .take(17))

['Every',
 'person',
 'had',
 'a',
 'star,',
 'every',
 'star',
 'had',
 'a',
 'friend,',
 'and',
 'for',
 'every',
 'person',
 'carrying',
 'a',
 'star']

In [46]:
(simple_rdd.flatMap(lambda line: line.split(' '))
           .map(lambda word: word.replace(',', '').lower())
           .take(10))

['every', 'person', 'had', 'a', 'star', 'every', 'star', 'had', 'a', 'friend']

In [13]:
(simple_rdd.flatMap(lambda line: line.split(' '))
           .map(lambda word: word.replace(',', '').lower())
           .map(lambda word: (word, 1))
           .take(5))

[('every', 1), ('person', 1), ('had', 1), ('a', 1), ('star', 1)]

In [14]:
(simple_rdd.flatMap(lambda line: line.split(' '))
           .map(lambda word: word.replace(',', '').lower())
           .map(lambda word: (word, 1))
           .reduceByKey(lambda a,b: a+b)
           .collect())

[('person', 2),
 ('there', 1),
 ('was', 1),
 ('carried', 1),
 ('this', 1),
 ('like', 1),
 ('secret', 1),
 ('confidante', 1),
 ('in', 1),
 ('heart', 1),
 ('every', 3),
 ('had', 2),
 ('a', 4),
 ('star', 3),
 ('friend', 1),
 ('and', 2),
 ('for', 1),
 ('carrying', 1),
 ('someone', 1),
 ('else', 1),
 ('who', 1),
 ('reflected', 1),
 ('it', 1),
 ('everyone', 1),
 ('reflection', 1),
 ('the', 1)]

In [15]:
def count_freq(rdd):
  return (rdd.flatMap(lambda line: line.split(' '))
            .map(lambda word: word.replace(',', '').lower())
            .map(lambda word: (word, 1))
            .reduceByKey(lambda a,b: a+b)
            .sortBy(lambda a: a[1], False)
            .collect())

In [16]:
!wget http://www.scifiscripts.com/scripts/swd1_5-74.txt

--2024-01-31 12:11:50--  http://www.scifiscripts.com/scripts/swd1_5-74.txt
Resolving www.scifiscripts.com (www.scifiscripts.com)... 207.32.177.145
Connecting to www.scifiscripts.com (www.scifiscripts.com)|207.32.177.145|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 203125 (198K) [text/plain]
Saving to: ‘swd1_5-74.txt’

swd1_5-74.txt       100%[===================>] 198.36K  --.-KB/s    in 0.08s   

2024-01-31 12:11:50 (2.43 MB/s) - ‘swd1_5-74.txt’ saved [203125/203125]



In [17]:
sw = sc.textFile('swd1_5-74.txt')

In [18]:
sw.take(10)

['The Star Wars',
 'by',
 'George Lucas',
 '',
 '',
 '',
 'Rough Draft [First of four major screenplay drafts]',
 'Lucasfilm Ltd.',
 '5/74',
 '']

In [19]:
count_freq(sw)[: 100]

[('', 3543),
 ('the', 2731),
 ('and', 929),
 ('a', 897),
 ('of', 744),
 ('to', 702),
 ('general', 459),
 ('is', 393),
 ('in', 359),
 ('-', 330),
 ('starkiller', 302),
 ('his', 289),
 ('on', 274),
 ('he', 243),
 ('are', 209),
 ('you', 197),
 ('with', 196),
 ('they', 177),
 ('as', 162),
 ('two', 161),
 ('into', 153),
 ('for', 146),
 ('it', 145),
 ('be', 145),
 ('han', 143),
 ('at', 137),
 ('i', 134),
 ('from', 134),
 ('out', 129),
 ('by', 125),
 ('an', 122),
 ('whitsun', 122),
 ('have', 119),
 ('one', 114),
 ('aquilae', 114),
 ('your', 106),
 ('but', 101),
 ('all', 95),
 ('we', 94),
 ('threepio', 92),
 ('through', 84),
 ('back', 83),
 ('princess', 82),
 ('this', 80),
 ('artwo', 80),
 ('their', 79),
 ('will', 79),
 ('small', 77),
 ('him', 75),
 ('up', 74),
 ('get', 73),
 ('fortress', 72),
 ('them', 71),
 ('there', 69),
 ('imperial', 69),
 ('down', 67),
 ('space', 66),
 ('over', 66),
 ('large', 65),
 ("it's", 64),
 ('then', 63),
 ("i'm", 63),
 ('not', 59),
 ('young', 58),
 ('before', 57),


In [20]:
import random

NUM_SAMPLES = 10**7

def inside(p):
    x, y = random.random(), random.random()
    return x*x + y*y < 1

count = (sc.parallelize(range(0, NUM_SAMPLES))
           .filter(inside).count())

print('Pi is roughly {}'.format(4.0 * count / NUM_SAMPLES))

Pi is roughly 3.1417288


In [21]:
!wget https://jacobceles.github.io/knowledge_repo/colab_and_pyspark/cars.csv

--2024-01-31 13:03:17--  https://jacobceles.github.io/knowledge_repo/colab_and_pyspark/cars.csv
Resolving jacobceles.github.io (jacobceles.github.io)... 185.199.110.153, 185.199.111.153, 185.199.108.153, ...
Connecting to jacobceles.github.io (jacobceles.github.io)|185.199.110.153|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://jacobcelestine.com/knowledge_repo/colab_and_pyspark/cars.csv [following]
--2024-01-31 13:03:17--  https://jacobcelestine.com/knowledge_repo/colab_and_pyspark/cars.csv
Resolving jacobcelestine.com (jacobcelestine.com)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to jacobcelestine.com (jacobcelestine.com)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 22608 (22K) [text/csv]
Saving to: ‘cars.csv’

cars.csv            100%[===================>]  22.08K  --.-KB/s    in 0.001s  

2024-01-31 13:03:17 (20.2 MB/s) - ‘cars.csv’ saved [22608/22608]



In [32]:
cars = spark.read.csv('cars.csv', header=True, sep=";")
cars.show(5)

+--------------------+----+---------+------------+----------+------+------------+-----+------+
|                 Car| MPG|Cylinders|Displacement|Horsepower|Weight|Acceleration|Model|Origin|
+--------------------+----+---------+------------+----------+------+------------+-----+------+
|Chevrolet Chevell...|18.0|        8|       307.0|     130.0| 3504.|        12.0|   70|    US|
|   Buick Skylark 320|15.0|        8|       350.0|     165.0| 3693.|        11.5|   70|    US|
|  Plymouth Satellite|18.0|        8|       318.0|     150.0| 3436.|        11.0|   70|    US|
|       AMC Rebel SST|16.0|        8|       304.0|     150.0| 3433.|        12.0|   70|    US|
|         Ford Torino|17.0|        8|       302.0|     140.0| 3449.|        10.5|   70|    US|
+--------------------+----+---------+------------+----------+------+------------+-----+------+
only showing top 5 rows



In [37]:
cars.select('Cylinders').rdd.map(lambda c: (c, 1)).reduceByKey(lambda a, b: a+b).collect()

[(Row(Cylinders='8'), 108),
 (Row(Cylinders='4'), 207),
 (Row(Cylinders='6'), 84),
 (Row(Cylinders='3'), 4),
 (Row(Cylinders='5'), 3)]